# TP Séance 9 — Détection d'objets en temps réel avec YOLO

**Durée : 3h — Version étudiant**

Dans ce TP, vous allez :
1. Utiliser YOLOv8 pour détecter des objets sur une image
2. Écrire des fonctions utilitaires (filtrage, comptage, IoU, affichage) validées par `pytest`
3. Brancher YOLO sur votre webcam pour faire de la détection **temps réel**
4. Afficher un compteur d'objets dans une **zone d'intérêt** (ROI)
5. Fine-tuner un modèle YOLOv8 pour **compter les doigts** en temps réel
   (collecte webcam → entraînement → bounding box via MediaPipe + classification YOLOv8)

## Plan (3h)

| Partie | Duree | A implementer | Fourni |
|--------|-------|---------------|--------|
| 1. Setup & rappel YOLO | 20 min | — | installation, chargement du modele |
| 2. Inference sur image | 20 min | — | exploration des resultats YOLO |
| 3. Fonctions utilitaires | 60 min | `filter_detections`, `count_by_class`, `compute_iou`, `draw_detections` | tests `pytest` |
| 4. Webcam temps reel | 45 min | `run_colab_webcam_full` | `video_stream`, `take_photo` |
| 5. Compteur de zone (ROI) | 25 min | `box_center`, `point_in_box`, `count_in_roi` | `run_colab_webcam_roi` |
| 6. Telephone comme camera (bonus) | 10 min | — | — |
| 7. Fine-tuning compteur de doigts | bonus | — | collecte, entrainement, inference guidee |

> Les cellules marquees `# TODO` sont les seules a completer. Les cellules `%%ipytest` valident votre implementation automatiquement.

**Rendu** : notebook execute avec toutes les cellules de test vertes + 2-3 captures d'ecran de la detection en cours.

## Partie 1 — Setup et rappel YOLO

YOLO (*You Only Look Once*) est un détecteur d'objets **single-shot** : une seule passe forward sur l'image produit les bounding boxes, les classes et les scores de confiance. C'est ce qui le rend suffisamment rapide pour du temps réel (30+ FPS sur CPU modeste avec `yolov8n`).

**Concepts clés**
- **Bounding box** : `(x1, y1, x2, y2)` en pixels (format `xyxy`) ou `(cx, cy, w, h)` normalisé
- **Score de confiance** : probabilité qu'un objet soit dans la box
- **IoU** : aire_intersection / aire_union — mesure le chevauchement entre deux boxes
- **NMS** : Non-Maximum Suppression, élimine les boxes redondantes


In [ ]:
!pip install ultralytics opencv-python ipytest pytest matplotlib mediapipe --quiet


In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
import torch
import time
import requests
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt

import ipytest
ipytest.autoconfig()

print("Imports OK")


In [ ]:
# Choix du modele :
#   yolov8n.pt = nano   (3.2M parametres, mAP 37.3) — le plus rapide (defaut TP)
#   yolov8s.pt = small  (11M  parametres, mAP 44.9) — bon compromis
#   yolov8m.pt = medium (26M  parametres, mAP 50.2) — plus precis, si GPU
#
# yolov8n peut confondre des objets proches (bouteille / telephone / telecommande).
# Pour plus de precision avec un GPU, passez a "yolov8s.pt" ou "yolov8m.pt".
model = YOLO("yolov8n.pt")
CLASS_NAMES = model.names
print(f"Modele charge : {len(CLASS_NAMES)} classes COCO")
print(f"Exemples : {list(CLASS_NAMES.values())[:10]}")


## Partie 2 — Inférence sur une image (échauffement)

Avant d'attaquer le temps réel, vérifions que le modèle fonctionne et comprenons la structure des résultats renvoyés par Ultralytics.


In [ ]:
img_url = "https://ultralytics.com/images/bus.jpg"
img_path = "bus.jpg"

if not Path(img_path).exists():
    Path(img_path).write_bytes(requests.get(img_url).content)

img = cv2.imread(img_path)
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Image de test")
plt.show()


In [ ]:
results = model(img_path, verbose=False)
r = results[0]

annotated = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8, 6))
plt.imshow(annotated)
plt.axis("off")
plt.title(f"{len(r.boxes)} détections (annotation d'Ultralytics)")
plt.show()


In [ ]:
print("--- Structure des résultats ---")
print(f"Nombre de boxes : {len(r.boxes)}")
print(f"Boxes xyxy (pixels) :\n{r.boxes.xyxy}\n")
print(f"Confiances : {r.boxes.conf}")
print(f"Classes    : {r.boxes.cls}")
print(f"Noms       : {[r.names[int(c)] for c in r.boxes.cls]}")


## Partie 3 — Fonctions utilitaires (testables)

Pour la boucle temps réel, on a besoin de **fonctions pures** qui opèrent sur les prédictions brutes de YOLO. On les écrit ici puis on les valide avec `pytest`.

> Les tests sont exécutés **dans le notebook** via `ipytest`. Une cellule qui commence par `%%ipytest` collecte toutes les fonctions `test_*` de la cellule et les lance comme `pytest` le ferait. **Ne modifiez pas les cellules de test.**


### 3.1 `filter_detections` — filtrer par confiance et par classe

Écrire une fonction qui garde uniquement les détections :
- de confiance **supérieure ou égale** à `conf_thresh`
- **et** dont la classe appartient à `target_classes` (si `target_classes is None`, on garde toutes les classes)

**Signature :**
```python
def filter_detections(boxes, scores, classes, conf_thresh=0.5, target_classes=None):
    """
    boxes          : tensor (N, 4) — format xyxy
    scores         : tensor (N,)
    classes        : tensor (N,)   — class ids (int)
    conf_thresh    : float
    target_classes : iterable[int] ou None
    Retourne (boxes_filt, scores_filt, classes_filt) — tensors du même type qu'en entrée.
    """
```

**Indice** : `torch.isin(classes, target_tensor)` crée un mask booléen.


In [ ]:
def filter_detections(boxes, scores, classes, conf_thresh=0.5, target_classes=None):
    """
    Filtre les detections par seuil de confiance et optionnellement par classe.

    Parametres
    ----------
    boxes         : Tensor (N, 4)  — coordonnees xyxy
    scores        : Tensor (N,)    — confiances
    classes       : Tensor (N,)    — identifiants de classe
    conf_thresh   : float          — seuil minimum de confiance (inclus)
    target_classes: liste ou None  — classes a garder (None = toutes)

    Retourne
    --------
    boxes_f, scores_f, classes_f : tensors filtres
    """
    # TODO 1 : construire un masque booleen -> scores >= conf_thresh
    # TODO 2 : si target_classes n'est pas None, affiner le masque
    #           (indice : torch.isin)
    # TODO 3 : retourner boxes[mask], scores[mask], classes[mask]
    raise NotImplementedError


In [ ]:
%%ipytest -v

def test_filter_by_confidence():
    boxes = torch.tensor([[0,0,10,10], [0,0,20,20], [0,0,30,30]], dtype=torch.float32)
    scores = torch.tensor([0.9, 0.3, 0.7])
    classes = torch.tensor([0, 1, 0])
    fb, fs, fc = filter_detections(boxes, scores, classes, conf_thresh=0.5)
    assert len(fb) == 2
    assert torch.allclose(fs, torch.tensor([0.9, 0.7]))

def test_filter_by_class():
    boxes = torch.tensor([[0,0,10,10], [0,0,20,20], [0,0,30,30]], dtype=torch.float32)
    scores = torch.tensor([0.9, 0.8, 0.7])
    classes = torch.tensor([0, 1, 2])
    fb, fs, fc = filter_detections(
        boxes, scores, classes, conf_thresh=0.0, target_classes=[1, 2]
    )
    assert len(fb) == 2
    assert fc.tolist() == [1, 2]

def test_filter_combined():
    boxes = torch.zeros((4, 4))
    scores = torch.tensor([0.9, 0.2, 0.8, 0.6])
    classes = torch.tensor([0, 0, 1, 1])
    fb, fs, fc = filter_detections(
        boxes, scores, classes, conf_thresh=0.5, target_classes=[0]
    )
    assert len(fb) == 1
    assert fc.tolist() == [0]

def test_filter_empty_result():
    boxes = torch.tensor([[0,0,10,10]], dtype=torch.float32)
    scores = torch.tensor([0.3])
    classes = torch.tensor([0])
    fb, fs, fc = filter_detections(boxes, scores, classes, conf_thresh=0.9)
    assert len(fb) == 0

def test_filter_preserves_dtype():
    boxes = torch.tensor([[0,0,10,10], [0,0,20,20]], dtype=torch.float32)
    scores = torch.tensor([0.9, 0.8])
    classes = torch.tensor([0, 1])
    fb, fs, fc = filter_detections(boxes, scores, classes, conf_thresh=0.5)
    assert fb.dtype == boxes.dtype
    assert fc.dtype == classes.dtype


### 3.2 `count_by_class` — compter par classe

Écrire une fonction qui renvoie un dictionnaire `{nom: nombre}` à partir d'une liste/tensor de `class_ids`.

**Signature :**
```python
def count_by_class(classes, names):
    """
    classes : tensor ou list[int] — class ids
    names   : dict {id: nom}
    Retourne : dict {nom: count}
    """
```


In [ ]:
def count_by_class(classes, names):
    """
    Compte le nombre de detections par classe.

    Parametres
    ----------
    classes : Tensor ou liste d'identifiants de classe
    names   : dict {class_id: nom}

    Retourne
    --------
    dict {nom: nombre}   ex: {"person": 3, "bus": 1}
    """
    # TODO 1 : convertir classes en liste d'entiers si necessaire
    #           (indice : hasattr(classes, 'tolist'))
    # TODO 2 : utiliser Counter pour compter par nom (r.names[c])
    # TODO 3 : retourner le dictionnaire
    raise NotImplementedError


In [ ]:
%%ipytest -v

def test_count_basic():
    classes = torch.tensor([0, 0, 0, 5])
    names = {0: "person", 5: "bus"}
    result = count_by_class(classes, names)
    assert result == {"person": 3, "bus": 1}

def test_count_empty():
    result = count_by_class(torch.tensor([], dtype=torch.int64), {0: "person"})
    assert result == {}

def test_count_from_list():
    result = count_by_class([0, 1, 1, 0, 1], {0: "cat", 1: "dog"})
    assert result == {"cat": 2, "dog": 3}

def test_count_single_class():
    result = count_by_class(torch.tensor([7, 7, 7]), {7: "car"})
    assert result == {"car": 3}


### 3.3 `compute_iou` — intersection over union

L'IoU mesure le chevauchement entre deux bounding boxes :

$$\text{IoU}(A, B) = \frac{\text{aire}(A \cap B)}{\text{aire}(A \cup B)}$$

**Signature :**
```python
def compute_iou(box_a, box_b):
    """
    box_a, box_b : tuple/list/array (x1, y1, x2, y2) en pixels
    Retourne : float entre 0 et 1 (0.0 si pas d'intersection ou union nulle)
    """
```

**Indice** : l'intersection a pour coordonnées
`(max(ax1,bx1), max(ay1,by1), min(ax2,bx2), min(ay2,by2))`
et son aire est nulle si `min - max < 0`.


In [ ]:
def compute_iou(box_a, box_b):
    """
    Calcule l'IoU (Intersection over Union) entre deux bounding boxes.

    Chaque box est un tuple (x1, y1, x2, y2) en pixels.
    Retourne un flottant entre 0.0 (aucun chevauchement) et 1.0 (boites identiques).

    Rappel :
        IoU = aire(A inter B) / aire(A union B)
        avec  union = aire_A + aire_B - inter
    """
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    # TODO 1 : calculer les coordonnees de l'intersection
    #           inter_x1 = max(ax1, bx1), inter_x2 = min(ax2, bx2), etc.
    # TODO 2 : calculer l'aire de l'intersection
    #           inter_w = max(0, inter_x2 - inter_x1)
    # TODO 3 : calculer les aires de A et B
    # TODO 4 : calculer union et retourner inter_area / union
    #           retourner 0.0 si union <= 0
    raise NotImplementedError


In [ ]:
%%ipytest -v

def test_iou_identical():
    box = (10, 10, 20, 20)
    assert compute_iou(box, box) == 1.0

def test_iou_no_overlap():
    assert compute_iou((0, 0, 10, 10), (20, 20, 30, 30)) == 0.0

def test_iou_no_overlap_touching():
    # Les boxes se touchent mais ne se chevauchent pas
    assert compute_iou((0, 0, 10, 10), (10, 0, 20, 10)) == 0.0

def test_iou_half_overlap():
    # A (10x10=100), B (10x10=100), intersection (5x10=50)
    # IoU = 50 / (100+100-50) = 50/150 = 1/3
    iou = compute_iou((0, 0, 10, 10), (5, 0, 15, 10))
    assert abs(iou - 1/3) < 1e-6

def test_iou_contained():
    # B (10x10=100) entierement dans A (20x20=400)
    # IoU = 100 / 400
    iou = compute_iou((0, 0, 20, 20), (5, 5, 15, 15))
    assert abs(iou - 0.25) < 1e-6

def test_iou_degenerate_box():
    # box de surface nulle
    assert compute_iou((5, 5, 5, 5), (0, 0, 10, 10)) == 0.0


### 3.4 `draw_detections` — dessiner les boxes

Écrire une fonction qui dessine les bounding boxes et leurs labels sur une image OpenCV **sans modifier l'originale**.

**Signature :**
```python
def draw_detections(frame, boxes, scores, classes, names, color=(0, 255, 0)):
    """
    frame   : ndarray BGR (H, W, 3)
    boxes   : tensor (N, 4) xyxy
    scores  : tensor (N,)
    classes : tensor (N,) class ids
    names   : dict {id: nom}
    color   : BGR tuple
    Retourne : nouvelle frame annotée (ndarray, meme shape et dtype).
    """
```

Pour chaque détection :
- dessiner un rectangle `cv2.rectangle(out, (x1,y1), (x2,y2), color, 2)`
- afficher le label `"<nom> <score:.2f>"` au-dessus de la box (`cv2.putText`)


In [ ]:
def draw_detections(frame, boxes, scores, classes, names, color=(0, 255, 0)):
    """
    Dessine les bounding boxes et leurs labels sur une copie du frame.

    Contraintes :
    - Ne pas modifier frame (travailler sur out = frame.copy())
    - Convertir boxes/scores/classes en numpy si ce sont des tensors
    - Afficher pour chaque detection : rectangle + label "nom conf"
    - Retourner l'image annotee
    """
    # TODO 1 : out = frame.copy()
    # TODO 2 : convertir boxes, scores, classes en numpy si necessaire
    #           (indice : if hasattr(x, 'cpu'): x = x.cpu().numpy())
    # TODO 3 : pour chaque (box, score, cls) :
    #   - x1, y1, x2, y2 = [int(v) for v in box]
    #   - label = f"{names[int(cls)]} {float(score):.2f}"
    #   - cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
    #   - cv2.putText(out, label, (x1, y1 - 5), ...)
    # TODO 4 : retourner out
    raise NotImplementedError


In [ ]:
%%ipytest -v

def test_draw_returns_copy():
    frame = np.zeros((100, 100, 3), dtype=np.uint8)
    boxes = torch.tensor([[10, 10, 50, 50]], dtype=torch.float32)
    scores = torch.tensor([0.9])
    classes = torch.tensor([0])
    out = draw_detections(frame, boxes, scores, classes, {0: "test"})
    assert frame.sum() == 0, "L'image originale ne doit pas etre modifiee"
    assert out.sum() > 0, "La sortie doit contenir des pixels dessines"
    assert out.shape == frame.shape
    assert out.dtype == np.uint8

def test_draw_empty_detections():
    frame = np.full((50, 50, 3), 128, dtype=np.uint8)
    out = draw_detections(
        frame,
        torch.zeros((0, 4)),
        torch.zeros(0),
        torch.zeros(0, dtype=torch.int64),
        {0: "test"},
    )
    assert np.array_equal(out, frame), "Sans detection, la frame doit etre inchangee"

def test_draw_box_color_applied():
    frame = np.zeros((200, 200, 3), dtype=np.uint8)
    boxes = torch.tensor([[50, 80, 150, 180]], dtype=torch.float32)
    scores = torch.tensor([0.8])
    classes = torch.tensor([0])
    out = draw_detections(frame, boxes, scores, classes, {0: "obj"}, color=(0, 255, 0))
    # Bord bas de la box : y=180, x in [50..150]. Le canal vert (index 1) doit etre > 0
    bottom_edge = out[180, 50:150, 1]
    assert bottom_edge.max() > 0, "Le rectangle doit etre dessine en vert"

def test_draw_preserves_shape_and_dtype():
    frame = np.random.randint(0, 255, (120, 160, 3), dtype=np.uint8)
    boxes = torch.tensor([[0, 0, 30, 30]], dtype=torch.float32)
    scores = torch.tensor([0.5])
    classes = torch.tensor([0])
    out = draw_detections(frame, boxes, scores, classes, {0: "x"})
    assert out.shape == frame.shape
    assert out.dtype == frame.dtype


### 3.5 Mise à l'épreuve sur l'image de test

On enchaîne les 4 fonctions sur les résultats YOLO de `bus.jpg` pour vérifier qu'elles collaborent bien.


In [ ]:
r = model(img_path, verbose=False)[0]

# Ne garder que les personnes (classe 0) avec conf >= 0.5
fb, fs, fc = filter_detections(
    r.boxes.xyxy, r.boxes.conf, r.boxes.cls,
    conf_thresh=0.5,
    target_classes=[0],
)

print(f"Détections filtrées : {len(fb)}")
print(f"Comptage : {count_by_class(fc, r.names)}")

annotated = draw_detections(cv2.imread(img_path), fb, fs, fc, r.names)
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Pipeline complet : personnes uniquement")
plt.show()


## Partie 4 — Détection temps réel avec votre webcam

> **Ce TP est conçu pour Google Colab.** La webcam y est capturée via **JavaScript** (`eval_js`),
> car `cv2.imshow` n'ouvre pas de fenêtre dans un notebook distant. On récupère chaque frame depuis
> le navigateur, on la passe dans Python (YOLO + helpers), puis on réaffiche l'image annotée.

La boucle de détection temps réel se résume à :
1. Démarrer le flux webcam (`video_stream()`)
2. Pour chaque frame :
   a. Récupérer la frame du navigateur (`eval_js('captureFrame()')`)
   b. Inférer avec YOLO
   c. Filtrer les détections (helper de la Partie 3)
   d. Dessiner (helper de la Partie 3)
   e. Réafficher l'image annotée
3. Arrêter avec le bouton **Arrêter le flux** (ou en interrompant le kernel)

> *En local* (hors Colab), on remplacerait `video_stream`/`eval_js` par `cv2.VideoCapture(0)` +
> `cv2.imshow`, avec sortie sur la touche **q** — non fourni ici.


### 4.1 Boucle basique


#### Google Colab : Accès Webcam via JavaScript

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capturer';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

try:
  filename = take_photo()
  print('Capture réussie ! Analyse avec YOLO...')

  # Inférence sur la photo capturée
  img = cv2.imread(filename)
  r = model(img, verbose=False)[0]
  annotated = r.plot()

  plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
  plt.axis('off')
  plt.show()
except Exception as err:
  print(str(err))

### 4.2 Version complète avec helpers et FPS

On réutilise `filter_detections`, `count_by_class` et `draw_detections`, et on affiche :
- le nombre d'objets par classe en haut à gauche
- les FPS lissés en haut à droite
- seulement certaines classes : `person` (0), `chair` (56), `cell phone` (67)


#### Google Colab (Stream Vidéo)

On capture le flux webcam via JavaScript (sandbox navigateur), on renvoie chaque frame au kernel Python, on passe dans YOLO + nos helpers, et on réaffiche l'image annotée.

> **Détection imprécise ?**
> YOLOv8**n** est très petit : il confond parfois bouteille / téléphone / télécommande. Deux leviers pour améliorer :
> 1. **Changer de modèle** — remplacer `yolov8n.pt` par `yolov8s.pt` dans la cellule de chargement (Partie 1)
> 2. **Lancer en `show_all=True`** pour voir *toutes* les classes que le modèle prédit — si on filtre sur `(0, 56, 67)` seulement, une bouteille correctement détectée (classe 39) est invisible et on ne voit que les erreurs.

Indices COCO utiles : `0=person  39=bottle  41=cup  56=chair  62=tv  63=laptop  64=mouse  65=remote  66=keyboard  67=cell phone  73=book`.


In [ ]:
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
import base64


def video_stream():
    js = Javascript('''
    var video;
    var div = null;
    var stream;
    var captureCanvas;
    var shutdown = false;

    function removeDom() {
        try { if (stream) stream.getTracks().forEach(t => t.stop()); } catch(e) {}
        try { if (div) div.remove(); } catch(e) {}
        div = null;
        stream = null;
    }

    function createDom() {
        div = document.createElement('div');
        div.style.display = 'flex';
        div.style.flexDirection = 'column';
        div.style.alignItems = 'center';

        var btn = document.createElement('button');
        btn.textContent = 'Arreter le flux';
        div.appendChild(btn);

        video = document.createElement('video');
        video.style.display = 'block';
        video.width = 640;
        video.height = 480;
        div.appendChild(video);

        document.body.appendChild(div);
        btn.onclick = () => { shutdown = true; };
    }

    async function startStream() {
        shutdown = false;
        createDom();
        stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
        video.srcObject = stream;
        await video.play();
        captureCanvas = document.createElement('canvas');
        captureCanvas.width = 640;
        captureCanvas.height = 480;
    }

    async function captureFrame() {
        if (shutdown) {
            removeDom();
            return null;
        }
        captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
        return captureCanvas.toDataURL('image/jpeg', 0.8);
    }

    window.startStream   = startStream;
    window.captureFrame  = captureFrame;
    window.removeDom     = removeDom;
    ''')
    display(js)
    eval_js('startStream()')


def run_colab_webcam(conf=0.5, target_classes=(0, 39, 41, 56, 67), show_all=False):
    """
    conf           : seuil de confiance (0.5 strict, 0.3 permissif)
    target_classes : classes a garder. Par defaut :
                       0 person, 39 bottle, 41 cup, 56 chair, 67 cell phone
    show_all       : True -> ignore target_classes et affiche TOUTES les detections
                     (utile pour diagnostiquer les mauvais classements)
    """
    video_stream()
    display_handle = display(None, display_id=True)
    prev_t = time.perf_counter()
    fps_smooth = 0.0

    try:
        while True:
            data = eval_js('captureFrame()')
            if data is None:
                break

            binary = base64.b64decode(data.split(',')[1])
            frame = cv2.imdecode(np.frombuffer(binary, np.uint8), cv2.IMREAD_COLOR)
            if frame is None:
                continue

            r = model(frame, conf=conf, verbose=False)[0]

            if show_all:
                fb, fs, fc = r.boxes.xyxy, r.boxes.conf, r.boxes.cls
            else:
                fb, fs, fc = filter_detections(
                    r.boxes.xyxy, r.boxes.conf, r.boxes.cls,
                    conf_thresh=conf, target_classes=target_classes,
                )

            annotated = draw_detections(frame, fb, fs, fc, r.names)

            counts = count_by_class(fc, r.names)
            y = 25
            for name, n in counts.items():
                cv2.putText(
                    annotated, f"{name}: {n}", (10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2,
                )
                y += 22

            now = time.perf_counter()
            inst_fps = 1.0 / max(now - prev_t, 1e-6)
            prev_t = now
            fps_smooth = 0.9 * fps_smooth + 0.1 * inst_fps
            cv2.putText(
                annotated, f"FPS: {fps_smooth:.1f}",
                (annotated.shape[1] - 160, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2,
            )

            _, encoded = cv2.imencode('.jpg', annotated)
            display_handle.update(Image(data=encoded.tobytes()))
    except Exception as e:
        print(f"Erreur: {e}")
    finally:
        try:
            eval_js('removeDom()')
        except Exception:
            pass


# Mode DEBUG : toutes les classes COCO visibles — utile pour verifier
# ce que le modele predit reellement.
run_colab_webcam(conf=0.5, show_all=True)

# Une fois que vous voyez que le modele predit bien, repassez en mode filtre :
# run_colab_webcam(conf=0.5, target_classes=(0, 39, 41, 56, 67))


In [ ]:
def run_colab_webcam_full(conf=0.4, target_classes=(0, 56, 67)):
    """
    Detection temps reel sur Colab avec helpers et affichage des FPS.

    Etapes :
    1. Demarrer le flux video avec video_stream()
    2. Creer un display_handle avec display(None, display_id=True)
    3. Boucle infinie :
       a. Recuperer une frame via eval_js('captureFrame()')
       b. Decoder la frame (base64 -> numpy BGR)
       c. Passer dans model(frame, ...) et extraire boxes/conf/cls
       d. Appeler filter_detections, count_by_class, draw_detections
       e. Afficher les FPS et le comptage par classe sur l'image
       f. Encoder en JPEG et mettre a jour display_handle
    4. Fermer avec eval_js('removeDom()') dans un bloc finally
    """
    # TODO
    raise NotImplementedError


run_colab_webcam_full()

## Partie 5 — Compteur dans une zone d'intérêt (ROI)

On définit un rectangle fixe sur l'image et on compte combien d'objets (du type recherché) ont le **centre de leur bbox à l'intérieur** de cette zone.

**Cas d'usage** : compter les personnes devant une porte, les voitures garées dans un emplacement, etc.


### 5.1 Helpers géométriques

Trois petites fonctions à écrire :
- `box_center(box)` → renvoie `(cx, cy)` le centre de la box
- `point_in_box(point, box)` → `True` si le point est à l'intérieur (bords inclus)
- `count_in_roi(boxes, roi)` → nombre de boxes dont le centre est dans `roi`


In [ ]:
def box_center(box):
    """Retourne (cx, cy) le centre de la bounding box (x1, y1, x2, y2)."""
    # TODO
    raise NotImplementedError


def point_in_box(point, box):
    """Retourne True si le point (px, py) est a l'interieur de la box (bords inclus)."""
    # TODO
    raise NotImplementedError


def count_in_roi(boxes, roi):
    """
    Compte combien de boxes ont leur centre dans la zone roi.

    boxes : Tensor ou array (N, 4)
    roi   : tuple (x1, y1, x2, y2)
    """
    # TODO 1 : convertir boxes en numpy si necessaire
    # TODO 2 : pour chaque box, verifier si box_center(b) est dans roi
    # TODO 3 : retourner le compte (int)
    raise NotImplementedError


In [ ]:
%%ipytest -v

def test_box_center_basic():
    assert box_center((0, 0, 10, 20)) == (5, 10)

def test_box_center_offset():
    assert box_center((10, 20, 30, 40)) == (20, 30)

def test_point_in_box_inside():
    assert point_in_box((5, 5), (0, 0, 10, 10)) is True

def test_point_in_box_outside():
    assert point_in_box((15, 5), (0, 0, 10, 10)) is False

def test_point_on_border():
    assert point_in_box((0, 0), (0, 0, 10, 10)) is True
    assert point_in_box((10, 10), (0, 0, 10, 10)) is True

def test_count_in_roi_mixed():
    boxes = torch.tensor([
        [0, 0, 10, 10],         # center (5, 5)     -> inside
        [100, 100, 120, 120],   # center (110, 110) -> outside
        [40, 40, 60, 60],       # center (50, 50)   -> inside
    ], dtype=torch.float32)
    assert count_in_roi(boxes, (0, 0, 80, 80)) == 2

def test_count_in_roi_empty():
    assert count_in_roi(torch.zeros((0, 4)), (0, 0, 10, 10)) == 0


### 5.2 Boucle webcam avec ROI

La ROI est un rectangle centré qui couvre 50% de la largeur et 60% de la hauteur de l'image. Elle passe en rouge dès qu'un objet ciblé entre dedans.


#### Google Colab

Cette cellule adapte la détection avec zone d'intérêt (ROI) pour le navigateur.

In [ ]:
def run_colab_webcam_roi(conf=0.5, target_classes=(0, 39, 41)):
    """
    Compteur de zone d'interet sur Colab.
    Par defaut, on compte : person, bottle, cup dans la ROI centrale.
    """
    video_stream()
    display_handle = display(None, display_id=True)

    w, h = 640, 480
    roi = (int(0.25 * w), int(0.2 * h), int(0.75 * w), int(0.8 * h))

    try:
        while True:
            data = eval_js('captureFrame()')
            if data is None:
                break

            binary = base64.b64decode(data.split(',')[1])
            frame = cv2.imdecode(np.frombuffer(binary, np.uint8), cv2.IMREAD_COLOR)
            if frame is None:
                continue

            r = model(frame, conf=conf, verbose=False)[0]

            fb, fs, fc = filter_detections(
                r.boxes.xyxy, r.boxes.conf, r.boxes.cls,
                conf_thresh=conf, target_classes=target_classes,
            )
            annotated = draw_detections(frame, fb, fs, fc, r.names)

            n_in = count_in_roi(fb, roi)
            color = (0, 0, 255) if n_in > 0 else (200, 200, 200)
            cv2.rectangle(annotated, (roi[0], roi[1]), (roi[2], roi[3]), color, 2)
            cv2.putText(
                annotated, f"ROI count: {n_in}", (roi[0], roi[1] - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2,
            )

            _, encoded = cv2.imencode('.jpg', annotated)
            display_handle.update(Image(data=encoded.tobytes()))
    except Exception as e:
        print(f"Erreur: {e}")
    finally:
        try:
            eval_js('removeDom()')
        except Exception:
            pass


run_colab_webcam_roi(conf=0.5, target_classes=(0, 39, 41))


## Partie 7 — Fine-tuning (pour info)

Le modèle COCO connaît 80 classes. Pour en reconnaître d'autres (badge d'école, logo, pièce mécanique), il faut **fine-tuner** :

```python
custom = YOLO("yolov8n.pt")
custom.train(
    data="mon_dataset/data.yaml",
    epochs=30,
    imgsz=416,
)
```

Le dataset doit suivre le **format YOLO** :
```
mon_dataset/
  images/train/*.jpg
  images/val/*.jpg
  labels/train/*.txt   # chaque ligne: class_id cx cy w h (normalises)
  labels/val/*.txt
  data.yaml            # path, train, val, nc, names
```

Des outils comme [Roboflow](https://roboflow.com) ou [LabelImg](https://github.com/HumanSignal/labelImg) permettent d'annoter rapidement. Ce sera le sujet de la **Séance 10** — aujourd'hui, profitez du temps réel !


### 7.1 Objectif : reconnaître le nombre de doigts levés

On combine deux modèles pour avoir à la fois une **bounding box** et un **comptage** :

| Etape | Modele | Rôle |
|-------|--------|------|
| 1 | **MediaPipe Hands** | Détecte la main et fournit la bounding box (pas d'entraînement) |
| 2 | **YOLOv8-cls** (fine-tuné) | Classifie le crop de la main en 0–5 doigts |

**Pourquoi cette approche ?**  
- MediaPipe localise la main avec précision sans aucune annotation.  
- YOLOv8-cls apprend à *compter* sur vos propres images collectées à la section 7.2.  
- On obtient : bounding box + label *N doigt(s)* + confiance, en temps réel.

### 7.2 Collecter ses propres images via la webcam

La cellule suivante ouvre la webcam et enregistre des frames dans `finger_dataset/train/<classe>/`.

- Appuyez sur **0 à 5** pour sauvegarder la frame courante dans la classe correspondante.
- Appuyez sur **v** pour basculer en mode `val` (validation) et collecter des images de val.
- Appuyez sur **q** pour quitter.

> Visez **50–100 images par classe** pour un premier entraînement exploitable.

In [ ]:
import cv2
import base64
import numpy as np
from pathlib import Path
from datetime import datetime
from IPython.display import display, Image
from google.colab.output import eval_js

DATASET_ROOT = Path("finger_dataset")
CLASSES = list(range(6))

for split in ["train", "val"]:
    for c in CLASSES:
        (DATASET_ROOT / split / str(c)).mkdir(parents=True, exist_ok=True)


def collect_finger_data_colab(class_id: int, n_images: int = 60, split: str = "train"):
    """
    Collecte n_images images de la classe class_id via la webcam Colab.
    Appelez cette fonction une fois par classe (0 a 5).
    """
    if class_id not in CLASSES:
        print(f"class_id doit etre compris entre 0 et 5, recu : {class_id}")
        return

    save_dir = DATASET_ROOT / split / str(class_id)
    save_dir.mkdir(parents=True, exist_ok=True)

    video_stream()
    display_handle = display(None, display_id=True)
    count = 0

    try:
        while count < n_images:
            data = eval_js('captureFrame()')
            if data is None:
                break

            binary = base64.b64decode(data.split(',')[1])
            frame = cv2.imdecode(np.frombuffer(binary, np.uint8), cv2.IMREAD_COLOR)
            if frame is None:
                continue

            ts = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
            fname = save_dir / f"finger_{class_id}_{ts}.jpg"
            cv2.imwrite(str(fname), frame)
            count += 1

            annotated = frame.copy()
            cv2.putText(
                annotated,
                f"Classe : {class_id} doigt(s)",
                (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 200, 255), 2,
            )
            cv2.putText(
                annotated,
                f"Capture : {count} / {n_images}",
                (10, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 200, 0), 2,
            )
            bar_w = int((count / n_images) * (annotated.shape[1] - 20))
            cv2.rectangle(annotated, (10, 80), (annotated.shape[1] - 10, 100), (60, 60, 60), -1)
            cv2.rectangle(annotated, (10, 80), (10 + bar_w, 100), (0, 200, 0), -1)

            _, encoded = cv2.imencode('.jpg', annotated)
            display_handle.update(Image(data=encoded.tobytes()))

    except Exception as e:
        print(f"Erreur : {e}")
    finally:
        try:
            eval_js('removeDom()')
        except Exception:
            pass

    print(f"Classe {class_id} ({split}) : {count} images sauvegardees dans {save_dir}")


# Changer class_id (0 a 5) et relancer la cellule pour chaque classe
collect_finger_data_colab(class_id=0, n_images=60, split="train")

### 7.3 Alternative : télécharger un dataset annoté (Roboflow)

Si vous ne souhaitez pas collecter vos propres images, des datasets publics de comptage de doigts sont disponibles sur [Roboflow Universe](https://universe.roboflow.com) (rechercher *finger counting* > filtre *Classification*).

```python
# Exemple — remplacer les placeholders par vos valeurs Roboflow
# !pip install roboflow --quiet
# from roboflow import Roboflow
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("WORKSPACE").project("finger-counting")
# dataset = project.version(1).download("folder", location="finger_dataset")
```

La cellule suivante vérifie la structure du dataset, quelle que soit la source.

In [ ]:
from pathlib import Path

DATASET_ROOT = Path("finger_dataset")


def show_dataset_stats(root: Path):
    """Affiche le nombre d'images par classe et par split."""
    for split in ["train", "val"]:
        split_path = root / split
        if not split_path.exists():
            print(f"[{split}] dossier absent")
            continue
        total = 0
        print(f"\n[{split}]")
        for cls_dir in sorted(split_path.iterdir()):
            if cls_dir.is_dir():
                imgs = list(cls_dir.glob("*.jpg")) + list(cls_dir.glob("*.png"))
                print(f"  Classe '{cls_dir.name}' : {len(imgs)} images")
                total += len(imgs)
        print(f"  Total : {total} images")


show_dataset_stats(DATASET_ROOT)

### 7.4 Fine-tuning du modèle de classification

On part de `yolov8n-cls.pt` (nano, pré-entraîné sur ImageNet) et on le spécialise sur 6 classes.

| Paramètre | Valeur | Rôle |
|-----------|--------|------|
| `epochs` | 30 | Nombre de passes sur le dataset |
| `imgsz` | 224 | Résolution d'entrée (carré) |
| `batch` | 16 | Taille du mini-batch |
| `lr0` | 0.001 | Taux d'apprentissage initial |
| `freeze` | 10 | Geler les 10 premières couches (transfer learning) |
| `patience` | 10 | Early stopping si pas d'amélioration pendant N époques |

In [ ]:
from ultralytics import YOLO
from pathlib import Path

DATASET_ROOT = Path("finger_dataset")

for split in ["train", "val"]:
    if not (DATASET_ROOT / split).exists():
        raise FileNotFoundError(
            f"Dossier manquant : {DATASET_ROOT / split}\n"
            "Lancez collect_finger_data() (section 7.2) ou telechargez le dataset (section 7.3)."
        )

model_cls = YOLO("yolov8n-cls.pt")

results_train = model_cls.train(
    data=str(DATASET_ROOT),
    epochs=30,
    imgsz=224,
    batch=16,
    lr0=0.001,
    freeze=10,
    patience=10,
    name="finger_counter",
    project="runs/classify",
    verbose=False,
)

print(f"Modele sauvegarde dans : {results_train.save_dir}/weights/best.pt")

### 7.5 Évaluation — accuracy et matrice de confusion

On mesure l'accuracy top-1 sur le split `val` et on affiche la matrice de confusion générée automatiquement par Ultralytics.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

best_weights = Path("runs/classify/finger_counter/weights/best.pt")

if not best_weights.exists():
    print("Modele introuvable. Lancez l'entrainement (section 7.4) d'abord.")
else:
    model_eval = YOLO(str(best_weights))
    metrics = model_eval.val(data=str(Path("finger_dataset")), verbose=False)

    print(f"Accuracy top-1 : {metrics.top1:.4f}")
    print(f"Accuracy top-5 : {metrics.top5:.4f}")

    conf_path = Path("runs/classify/finger_counter/confusion_matrix.png")
    if conf_path.exists():
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.imshow(mpimg.imread(str(conf_path)))
        ax.axis("off")
        ax.set_title("Matrice de confusion — Compteur de doigts")
        plt.tight_layout()
        plt.show()

### 7.6 Inférence en temps réel — compteur de doigts avec bounding box

**Pipeline par frame :**
1. MediaPipe détecte la/les main(s) → coordonnées des 21 landmarks
2. On calcule la bounding box à partir des landmarks (min/max x, y + padding)
3. On crop la main et on passe le crop dans le modèle YOLOv8-cls fine-tuné
4. On dessine la bounding box et le label *N doigt(s)* sur l'image complète

> Cliquez sur **Arreter le flux** ou interrompez le kernel pour quitter.

In [ ]:
import cv2
import base64
import time
import numpy as np
import urllib.request
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from ultralytics import YOLO
from pathlib import Path
from IPython.display import display, Image
from google.colab.output import eval_js

BEST_WEIGHTS = Path("runs/classify/finger_counter/weights/best.pt")

FINGER_COLORS = {
    "0": (50,  50,  200),
    "1": (50,  200,  50),
    "2": (200, 200,  50),
    "3": (200,  50, 200),
    "4": (50,  200, 200),
    "5": (200, 100,  50),
}

# Telechargement du modele MediaPipe Hands (une seule fois)
MODEL_PATH = Path("hand_landmarker.task")
if not MODEL_PATH.exists():
    print("Telechargement du modele MediaPipe Hands...")
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/"
        "hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task",
        str(MODEL_PATH),
    )
    print("Modele telecharge.")


def get_hand_bbox(landmarks, h, w, padding=20):
    """Calcule (x1, y1, x2, y2) en pixels depuis les landmarks MediaPipe."""
    xs = [lm.x * w for lm in landmarks]
    ys = [lm.y * h for lm in landmarks]
    x1 = max(0, int(min(xs)) - padding)
    y1 = max(0, int(min(ys)) - padding)
    x2 = min(w, int(max(xs)) + padding)
    y2 = min(h, int(max(ys)) + padding)
    return x1, y1, x2, y2


def run_finger_counter(conf_thresh=0.35):
    """
    Boucle temps reel : MediaPipe Tasks detecte la main (bbox),
    YOLOv8-cls compte les doigts sur le crop.
    """
    if not BEST_WEIGHTS.exists():
        print(f"Modele introuvable : {BEST_WEIGHTS}")
        print("Entrainez le modele (section 7.4) d'abord.")
        return

    model_ft = YOLO(str(BEST_WEIGHTS))
    class_names = model_ft.names
    print(f"Modele charge — classes : {class_names}")

    base_options = mp_python.BaseOptions(model_asset_path=str(MODEL_PATH))
    options = mp_vision.HandLandmarkerOptions(
        base_options=base_options,
        running_mode=mp_vision.RunningMode.IMAGE,
        num_hands=2,
        min_hand_detection_confidence=0.5,
    )
    landmarker = mp_vision.HandLandmarker.create_from_options(options)

    video_stream()
    display_handle = display(None, display_id=True)
    prev_time = time.time()

    try:
        while True:
            data = eval_js('captureFrame()')
            if data is None:
                break

            binary = base64.b64decode(data.split(',')[1])
            frame = cv2.imdecode(np.frombuffer(binary, np.uint8), cv2.IMREAD_COLOR)
            if frame is None:
                continue

            h, w = frame.shape[:2]
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
            detection = landmarker.detect(mp_image)

            annotated = frame.copy()
            now = time.time()
            fps = 1.0 / max(now - prev_time, 1e-6)
            prev_time = now

            if detection.hand_landmarks:
                for hand_landmarks in detection.hand_landmarks:
                    x1, y1, x2, y2 = get_hand_bbox(hand_landmarks, h, w)
                    crop = frame[y1:y2, x1:x2]
                    if crop.size == 0:
                        continue

                    probs = model_ft(crop, verbose=False)[0].probs
                    top_idx = int(probs.top1)
                    top_conf = float(probs.top1conf)
                    class_name = class_names[top_idx]
                    color = FINGER_COLORS.get(class_name, (255, 255, 255))
                    draw_color = color if top_conf >= conf_thresh else (120, 120, 120)

                    cv2.rectangle(annotated, (x1, y1), (x2, y2), draw_color, 2)
                    label = f"{class_name} doigt(s)  {top_conf:.2f}"
                    (tw, th), _ = cv2.getTextSize(
                        label, cv2.FONT_HERSHEY_SIMPLEX, 0.65, 2
                    )
                    cv2.rectangle(
                        annotated,
                        (x1, y1 - th - 8), (x1 + tw + 4, y1),
                        draw_color, -1,
                    )
                    cv2.putText(
                        annotated, label, (x1 + 2, y1 - 4),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 0), 2,
                    )
            else:
                cv2.putText(
                    annotated, "Aucune main detectee",
                    (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (100, 100, 100), 2,
                )

            cv2.putText(
                annotated, f"FPS : {fps:.1f}", (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1,
            )
            _, encoded = cv2.imencode('.jpg', annotated)
            display_handle.update(Image(data=encoded.tobytes()))

    except Exception as e:
        print(f"Erreur : {e}")
    finally:
        landmarker.close()
        try:
            eval_js('removeDom()')
        except Exception:
            pass


run_finger_counter()


## Synthèse et livrables

- [ ] Toutes les cellules `%%ipytest` passent (vert)
- [ ] La boucle `run_webcam_full` a tourné sur votre machine (capture d'écran dans le rendu)
- [ ] Le compteur ROI `run_webcam_roi` fonctionne (capture d'écran)
- [ ] **Bonus** : test avec le téléphone

**Rendu** : notebook exécuté + 2 ou 3 captures d'écran (`.png`) de la détection en cours.

### Pour aller plus loin
- Ajouter un *tracker* (`model.track(...)` intégré à Ultralytics) pour compter les **passages uniques** dans la ROI
- Essayer `yolov8s.pt` ou `yolov8m.pt` → comparer les FPS
- Enregistrer un MP4 annoté avec `cv2.VideoWriter`
- Déclencher un son / une notification quand la ROI dépasse un seuil
